# Task 07: Interactive Periodic Noise Removal with Fourier

Nesta atividade você construirá um pipeline interativo para adicionar ruído senoidal e removê-lo com um filtro notch-reject no domínio da frequência.

**Pré-requisitos:** imagens NumPy grayscale, números complexos e conceitos iniciais de DFT.

## Roteiro

1. Criar ruído senoidal.
2. Adicionar o ruído sem clipping.
3. Inspecionar DFT e espectro centralizado.
4. Criar filtro notch-reject.
5. Restaurar por IDFT e comparar RMSE.
6. Explorar os parâmetros com sliders.

## 1. Importações

In [ ]:
import ipywidgets as widgets  # noqa: F401
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display  # noqa: F401

## 2. Imagem de entrada

Usaremos uma imagem grayscale sintética para que o efeito do ruído e da restauração seja fácil de observar.

In [ ]:
height, width = 128, 128
y, x = np.mgrid[:height, :width]
original_image = (60 + 0.7 * x + 0.3 * y).astype(np.float64)

plt.figure(figsize=(4, 4))
plt.imshow(original_image, cmap="gray")
plt.title("Imagem original")
plt.axis("off")
plt.show()

## 3. Criação do ruído senoidal

Uma senoide espacial gera dois impulsos simétricos no espectro.

Implemente:

`noise(y, x) = A * cos(2*pi*(frequency_x*x/width + frequency_y*y/height))`

onde `A` é a amplitude e `frequency_x` e `frequency_y` representam o número
de ciclos nas direções horizontal e vertical, respectivamente.

Retorne um array `float64` com o shape informado.

In [ ]:
def create_sinusoidal_noise(shape, amplitude, frequency_x, frequency_y):
    """Creates a 2D sinusoidal noise pattern with dtype float64."""
    noise = None

    ### START CODE HERE ###

    # TODO

    ### END CODE HERE ###

    return noise

## 4. Adição do ruído

Converta os cálculos para `float64`, preserve a imagem de entrada e adicione o
ruído diretamente à imagem.

Não aplique clipping para `[0, 255]` antes da análise no domínio da frequência.

In [ ]:
def add_periodic_noise(image, noise):
    """Adds periodic noise to a grayscale image without clipping."""
    noisy_image = None

    ### START CODE HERE ###

    # TODO

    ### END CODE HERE ###

    return noisy_image

## 5. Filtro notch-reject

Depois de `fftshift(fft2(noisy_image))`, o centro do espectro é:

`center_y = height // 2`  
`center_x = width // 2`

Os dois impulsos do ruído aparecem em:

`(center_y + frequency_y, center_x + frequency_x)`

e

`(center_y - frequency_y, center_x - frequency_x)`.

Crie inicialmente um filtro preenchido com `1.0`. Para cada notch, atribua
`0.0` aos pontos cuja distância ao centro do notch satisfaça
`distance <= radius`.

O componente DC no centro do espectro deve permanecer igual a `1.0`.

In [ ]:
def create_notch_reject_filter(shape, frequency_x, frequency_y, radius):
    """Creates an ideal notch-reject filter with dtype float64."""
    notch_filter = None

    ### START CODE HERE ###

    # TODO

    ### END CODE HERE ###

    return notch_filter

## 6. Restauração e RMSE

A restauração deve seguir explicitamente a sequência FFT, shift, multiplicação pelo filtro, inverse shift, inverse FFT e parte real. Não aplique clipping nem converta para `uint8`.

In [ ]:
def restore_image(noisy_image, notch_filter):
    """Restores an image using frequency-domain notch filtering."""
    restored = None

    ### START CODE HERE ###

    # TODO

    ### END CODE HERE ###

    return restored


def rmse(reference, result):
    """Computes the root mean squared error in float64."""
    value = None

    ### START CODE HERE ###

    # TODO

    ### END CODE HERE ###

    return value

## 7. Teste determinístico

Este caso usa uma imagem constante e uma frequência horizontal conhecida. Execute-o depois de implementar as funções anteriores.

In [ ]:
image = np.full((8, 8), 100, dtype=np.uint8)
original = image.copy()
amplitude = 20.0
frequency_x = 2
frequency_y = 0

noise = create_sinusoidal_noise(image.shape, amplitude, frequency_x, frequency_y)
expected_noise_row = np.array([20, 0, -20, 0, 20, 0, -20, 0], dtype=np.float64)
assert np.allclose(noise[0], expected_noise_row, atol=1e-12)

vertical_noise = create_sinusoidal_noise(
    (4, 4),
    amplitude=10.0,
    frequency_x=0,
    frequency_y=1,
)

expected_vertical = np.array(
    [10, 0, -10, 0],
    dtype=np.float64,
)

assert np.allclose(
    vertical_noise[:, 0],
    expected_vertical,
    atol=1e-12,
)

noisy_image = add_periodic_noise(image, noise)
expected_noisy_row = np.array([120, 100, 80, 100, 120, 100, 80, 100], dtype=np.float64)
assert np.allclose(noisy_image[0], expected_noisy_row, atol=1e-12)

notch_filter = create_notch_reject_filter(
    image.shape, frequency_x, frequency_y, radius=0
)
assert notch_filter[4, 2] == 0.0
assert notch_filter[4, 6] == 0.0
assert notch_filter[4, 4] == 1.0

notch_2d = create_notch_reject_filter(
    (8, 10),
    frequency_x=2,
    frequency_y=1,
    radius=0,
)

assert notch_2d[5, 7] == 0.0
assert notch_2d[3, 3] == 0.0
assert notch_2d[4, 5] == 1.0

notch_radius = create_notch_reject_filter(
    (9, 11),
    frequency_x=3,
    frequency_y=0,
    radius=1,
)

assert notch_radius[4, 8] == 0.0
assert notch_radius[3, 8] == 0.0
assert notch_radius[4, 7] == 0.0

# Diagonal está fora do círculo de raio 1
assert notch_radius[3, 7] == 1.0

restored = restore_image(noisy_image, notch_filter)
assert np.allclose(restored, image.astype(np.float64), atol=1e-10)
assert rmse(image, restored) < rmse(image, noisy_image)
assert rmse(image, restored) < 1e-10

for result in (noise, noisy_image, notch_filter, restored):
    assert result.shape == image.shape
    assert result.dtype == np.float64
assert np.array_equal(image, original)
print("Test passed!")

## 8. Interactive Experiment

A função abaixo atualiza todas as etapas e visualizações. Leia o pipeline e,
na próxima célula, use o slider de amplitude fornecido como exemplo para criar
os sliders de frequência X, frequência Y e raio do notch.

In [ ]:
def update_pipeline(amplitude, frequency_x, frequency_y, notch_radius):
    noise = create_sinusoidal_noise(
        original_image.shape, amplitude, frequency_x, frequency_y
    )
    noisy = add_periodic_noise(original_image, noise)
    spectrum = np.fft.fftshift(np.fft.fft2(noisy))
    magnitude = np.log1p(np.abs(spectrum))
    notch = create_notch_reject_filter(
        original_image.shape, frequency_x, frequency_y, notch_radius
    )
    restored = restore_image(noisy, notch)

    figure, axes = plt.subplots(2, 3, figsize=(13, 8))
    panels = (
        (original_image, "Original", "gray"),
        (noise, "Ruído senoidal", "gray"),
        (noisy, "Imagem com ruído", "gray"),
        (magnitude, "Espectro de magnitude", "magma"),
        (notch, "Filtro notch-reject", "gray"),
        (restored, "Imagem restaurada", "gray"),
    )
    for axis, (data, title, cmap) in zip(axes.ravel(), panels, strict=True):
        axis.imshow(data, cmap=cmap)
        axis.set_title(title)
        axis.axis("off")
    figure.tight_layout()
    plt.show()
    print(f"RMSE noisy: {rmse(original_image, noisy):.6f}")
    print(f"RMSE restored: {rmse(original_image, restored):.6f}")

In [ ]:
amplitude_slider = widgets.FloatSlider(
    value=20.0,
    min=0.0,
    max=100.0,
    step=5.0,
    description="Amplitude:",
    continuous_update=True,
)

frequency_x_slider = None
frequency_y_slider = None
notch_radius_slider = None

### START CODE HERE ###

# TODO: create frequency_x_slider, frequency_y_slider
# and notch_radius_slider following the example above.
# Keep continuous_update=True for interactive feedback.

### END CODE HERE ###

interactive_output = widgets.interactive_output(
    update_pipeline,
    {
        "amplitude": amplitude_slider,
        "frequency_x": frequency_x_slider,
        "frequency_y": frequency_y_slider,
        "notch_radius": notch_radius_slider,
    },
)

display(
    widgets.VBox(
        [
            amplitude_slider,
            frequency_x_slider,
            frequency_y_slider,
            notch_radius_slider,
        ]
    ),
    interactive_output,
)

## Explore

Experimente diferentes valores de amplitude, frequência X, frequência Y e raio do notch. Observe o padrão espacial do ruído, o deslocamento dos impulsos no espectro, o efeito do raio do filtro e os valores de RMSE antes e depois da restauração. Um raio muito grande pode remover informação útil.